# 스트레스 점수 예측 V12 — TabPFN Foundation Model

완전히 다른 계열의 사전학습 표형 데이터 모델을 실험합니다.

- TabPFNRegressor 5-Fold MAE 검증
- 기존 Public 최고 ExtraTrees와 동일 Fold에서 비교
- ExtraTrees + TabPFN 블렌딩 비율 자동 탐색
- Train에서만 모델과 전처리 기준 학습
- Google Colab GPU 런타임 권장

> Colab 메뉴에서 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택하세요.

## 1. TabPFN 설치

In [3]:
# 최신 API 변화를 피하기 위해 검증한 버전을 고정합니다.
!pip install -q "tabpfn==8.1.0" 

## 2. 최초 1회 라이선스 동의 및 인증

1. [Prior Labs UX](https://ux.priorlabs.ai)에 로그인하거나 가입합니다.
2. **Licenses** 탭에서 TabPFN 라이선스에 동의합니다.
3. [Account 페이지](https://ux.priorlabs.ai/account)에서 API Key를 복사합니다.
4. 아래 셀의 입력창에 Key를 붙여넣습니다.

API Key는 화면에 표시되지 않으며 노트북 파일에도 저장되지 않습니다.  
**토큰을 채팅이나 GitHub에 올리지 마세요.**

In [ ]:
import os
from getpass import getpass

if not os.environ.get("TABPFN_TOKEN"):
    token = getpass("TABPFN API Key를 붙여넣고 Enter: ").strip()
    if not token:
        raise ValueError("API Key가 입력되지 않았습니다.")
    os.environ["TABPFN_TOKEN"] = token

print("TabPFN 인증 토큰 설정 완료")

## 3. 라이브러리 및 GPU 확인

In [4]:
from pathlib import Path
import zipfile

import numpy as np
import pandas as pd
import torch
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from tabpfn import TabPFNRegressor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE != "cuda":
    raise RuntimeError("Colab 런타임을 T4 GPU로 변경해주세요.")

Device: cpu


RuntimeError: Colab 런타임을 T4 GPU로 변경해주세요.

## 4. 데이터 업로드

`Desktop/stress_project/open/open.zip` 파일을 업로드하세요.  
압축 안에는 `train.csv`, `test.csv`, `sample_submission.csv`가 있어야 합니다.

In [ ]:
from google.colab import files

uploaded = files.upload()
zip_candidates = [
    Path(name) for name in uploaded
    if name.lower().endswith(".zip")
]
if not zip_candidates:
    raise FileNotFoundError("open.zip 파일을 업로드해주세요.")

with zipfile.ZipFile(zip_candidates[0]) as archive:
    archive.extractall("/content/stress_data")

csv_files = list(Path("/content/stress_data").rglob("train.csv"))
if not csv_files:
    raise FileNotFoundError("압축 파일에서 train.csv를 찾지 못했습니다.")

DATA_DIR = csv_files[0].parent
print("Data directory:", DATA_DIR)

## 5. 설정 및 데이터 검증

In [ ]:
TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_ESTIMATORS = 1200
BEST_QUANTILE = 0.51

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

if set(train.columns) - {TARGET} != set(test.columns):
    raise ValueError("Train과 Test 입력 컬럼이 다릅니다.")
if list(submission.columns) != [ID_COLUMN, TARGET]:
    raise ValueError("제출 양식 컬럼이 올바르지 않습니다.")
if not submission[ID_COLUMN].equals(test[ID_COLUMN]):
    raise ValueError("Test와 제출 양식의 ID 순서가 다릅니다.")

print("Train:", train.shape)
print("Test:", test.shape)
train.head()

## 6. 행 단위 파생변수

In [ ]:
WEIGHTED_FEATURES = [
    "mean_working",
    "bmi",
    "cholesterol",
    "height",
    "glucose",
    "weight",
    "cholesterol_glucose_ratio",
    "bone_density",
]


def add_base_features(frame):
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = (
        result["cholesterol"] / (result["glucose"] + 1e-6)
    )
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_extratrees_features(frame):
    result = add_base_features(frame)
    for feature in WEIGHTED_FEATURES:
        result[f"{feature}__copy1"] = result[feature]
        result[f"{feature}__copy2"] = result[feature]
    return result


# TabPFN에는 중복 가중치 없이 자연스러운 행 단위 피처만 제공합니다.
X_tabpfn = add_base_features(
    train.drop(columns=[TARGET, ID_COLUMN])
)
X_test_tabpfn = add_base_features(
    test.drop(columns=[ID_COLUMN])
)

# 기존 최고 모델은 제출 당시와 동일하게 피처를 두 번 복제합니다.
X_extra = add_extratrees_features(
    train.drop(columns=[TARGET, ID_COLUMN])
)
X_test_extra = add_extratrees_features(
    test.drop(columns=[ID_COLUMN])
)
y = train[TARGET].to_numpy()

categorical_columns = X_tabpfn.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()
categorical_indices = [
    X_tabpfn.columns.get_loc(column)
    for column in categorical_columns
]

# TabPFN 내부 인코더가 범주형을 처리하도록 문자열 dtype을 유지합니다.
for column in categorical_columns:
    X_tabpfn[column] = (
        X_tabpfn[column].fillna("missing").astype(str)
    )
    X_test_tabpfn[column] = (
        X_test_tabpfn[column].fillna("missing").astype(str)
    )

print("TabPFN features:", X_tabpfn.shape)
print("Categorical columns:", categorical_columns)

## 7. 기존 최고 ExtraTrees 함수

In [ ]:
def make_extra_preprocessor(frame):
    categorical = frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()

    return ColumnTransformer(
        [
            (
                "categorical",
                Pipeline(
                    [
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="constant",
                                fill_value="missing",
                            ),
                        ),
                        (
                            "encoder",
                            OneHotEncoder(
                                handle_unknown="ignore",
                                sparse_output=False,
                            ),
                        ),
                    ]
                ),
                categorical,
            ),
            (
                "numerical",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                ),
                numerical,
            ),
        ]
    )


def extra_quantile_prediction(model, matrix):
    tree_predictions = np.column_stack(
        [tree.predict(matrix) for tree in model.estimators_]
    )
    return np.clip(
        np.quantile(
            tree_predictions,
            BEST_QUANTILE,
            axis=1,
        ),
        0.0,
        1.0,
    )

## 8. 동일한 5-Fold에서 TabPFN과 최고 모델 비교

첫 실행에서는 사전학습 가중치를 내려받기 때문에 시간이 더 걸릴 수 있습니다.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
oof_extra = np.zeros(len(train))
oof_tabpfn = np.zeros(len(train))

for fold, (train_idx, valid_idx) in enumerate(
    cv.split(X_tabpfn),
    start=1,
):
    # 기존 최고 ExtraTrees
    preprocessor = make_extra_preprocessor(
        X_extra.iloc[train_idx]
    )
    extra_train = preprocessor.fit_transform(
        X_extra.iloc[train_idx]
    )
    extra_valid = preprocessor.transform(
        X_extra.iloc[valid_idx]
    )
    extra_model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS,
        min_samples_leaf=1,
        max_features=1,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    extra_model.fit(extra_train, y[train_idx])
    oof_extra[valid_idx] = extra_quantile_prediction(
        extra_model,
        extra_valid,
    )

    # 사전학습 TabPFN
    tabpfn_model = TabPFNRegressor(
        device=DEVICE,
        random_state=RANDOM_SEED + fold,
        categorical_features_indices=categorical_indices,
    )
    tabpfn_model.fit(
        X_tabpfn.iloc[train_idx],
        y[train_idx],
    )
    tabpfn_prediction = tabpfn_model.predict(
        X_tabpfn.iloc[valid_idx]
    )
    oof_tabpfn[valid_idx] = np.clip(
        tabpfn_prediction,
        0.0,
        1.0,
    )

    extra_fold_mae = mean_absolute_error(
        y[valid_idx],
        oof_extra[valid_idx],
    )
    tabpfn_fold_mae = mean_absolute_error(
        y[valid_idx],
        oof_tabpfn[valid_idx],
    )
    print(
        f"Fold {fold} | "
        f"ExtraTrees={extra_fold_mae:.6f} | "
        f"TabPFN={tabpfn_fold_mae:.6f}"
    )

extra_cv_mae = mean_absolute_error(y, oof_extra)
tabpfn_cv_mae = mean_absolute_error(y, oof_tabpfn)

print("Best ExtraTrees CV MAE:", f"{extra_cv_mae:.6f}")
print("TabPFN CV MAE:", f"{tabpfn_cv_mae:.6f}")

## 9. 블렌딩 비율 탐색

In [ ]:
blend_results = []

for tabpfn_weight in np.arange(0.0, 0.51, 0.05):
    prediction = (
        (1.0 - tabpfn_weight) * oof_extra
        + tabpfn_weight * oof_tabpfn
    )
    raw_mae = mean_absolute_error(y, prediction)
    rounded_mae = mean_absolute_error(
        y,
        np.round(prediction, 2),
    )
    blend_results.append(
        (raw_mae, tabpfn_weight, False)
    )
    blend_results.append(
        (rounded_mae, tabpfn_weight, True)
    )

blend_results.sort()
results_frame = pd.DataFrame(
    [
        {
            "MAE": score,
            "ExtraTrees_weight": 1.0 - weight,
            "TabPFN_weight": weight,
            "round_2_decimals": rounded,
        }
        for score, weight, rounded in blend_results
    ]
)
results_frame.head(15)

## 10. 최종 모델 학습

블렌딩이 기존 최고 모델보다 좋아진 경우에만 TabPFN을 반영합니다.

In [ ]:
best_mae, selected_tabpfn_weight, use_rounding = blend_results[0]
selected_extra_weight = 1.0 - selected_tabpfn_weight

print("Selected CV MAE:", f"{best_mae:.6f}")
print("ExtraTrees weight:", selected_extra_weight)
print("TabPFN weight:", selected_tabpfn_weight)
print("Round to 2 decimals:", use_rounding)

# 전체 Train으로 기존 최고 모델 학습
final_preprocessor = make_extra_preprocessor(X_extra)
extra_train_full = final_preprocessor.fit_transform(X_extra)
extra_test_full = final_preprocessor.transform(X_test_extra)
final_extra = ExtraTreesRegressor(
    n_estimators=N_ESTIMATORS,
    min_samples_leaf=1,
    max_features=1,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
final_extra.fit(extra_train_full, y)
pred_extra = extra_quantile_prediction(
    final_extra,
    extra_test_full,
)

# 전체 Train으로 TabPFN 학습
final_tabpfn = TabPFNRegressor(
    device=DEVICE,
    random_state=RANDOM_SEED,
    categorical_features_indices=categorical_indices,
)
final_tabpfn.fit(X_tabpfn, y)
pred_tabpfn = np.clip(
    final_tabpfn.predict(X_test_tabpfn),
    0.0,
    1.0,
)

pred_v12 = (
    selected_extra_weight * pred_extra
    + selected_tabpfn_weight * pred_tabpfn
)
if use_rounding:
    pred_v12 = np.round(pred_v12, 2)
pred_v12 = np.clip(pred_v12, 0.0, 1.0)

print("Prediction range:", pred_v12.min(), pred_v12.max())
print("첫 10개 예측값:", pred_v12[:10])

## 11. 제출 파일 저장 및 다운로드

In [ ]:
submission_v12 = submission.copy()
submission_v12[TARGET] = pred_v12

OUTPUT_PATH = Path("/content/submit_v12_tabpfn_blend.csv")
submission_v12.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8",
)

print("Saved:", OUTPUT_PATH)
print("Rows:", len(submission_v12))
submission_v12.head(10)

In [ ]:
from google.colab import files
files.download(str(OUTPUT_PATH))